<a href="https://colab.research.google.com/github/thakurkavya11/ABES_WEBD_PA_1/blob/main/KAVYA_CHAUHAN_(2026B03110005).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# Project 17: Employee Attendance & HR Attrition Overview

In [44]:
import pandas as pd
import numpy as np

In [45]:
# ---------- 0. Column mapping (edit if your names differ) ----------
COL = dict(
    id="Employee_ID", dept="Department", level="Job_Level",
    income="Monthly_Income", sat="Satisfaction_Score",
    ot_hours="Overtime_Hours", ot_flag="Overtime",
    tenure="Tenure_Years", dist="Distance_From_Home", attr="Attrition"
)

In [61]:
# ---------- 1. Load & inspect ----------
df = pd.read_csv("17_hr_attrition.csv")
print(df.shape)
df.info()
print(df.head())
print(df.describe(include="all").T)

(1008, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Employee_ID         1008 non-null   object 
 1   Age                 1008 non-null   int64  
 2   Gender              1008 non-null   object 
 3   Department          1008 non-null   object 
 4   Job_Level           1008 non-null   object 
 5   Monthly_Income      978 non-null    float64
 6   Tenure_Years        1008 non-null   float64
 7   Distance_From_Home  1008 non-null   float64
 8   Overtime            1008 non-null   object 
 9   Overtime_Hours      1008 non-null   float64
 10  Satisfaction_Score  968 non-null    float64
 11  Attrition           1008 non-null   object 
dtypes: float64(5), int64(1), object(6)
memory usage: 94.6+ KB
  Employee_ID  Age  Gender Department Job_Level  Monthly_Income  Tenure_Years  \
0       E1113   49    Male        R&D   Manager     

In [47]:
# ---------- 2. Missing values & duplicates ----------
print(df.isna().sum())
print("Duplicate Employee_ID:", df.duplicated(subset=COL["id"]).sum())
df = df.drop_duplicates(subset=COL["id"], keep="first")

# Standardise Attrition to Yes/No text and a 0/1 flag
df[COL["attr"]] = df[COL["attr"]].astype(str).str.strip().str.title()
df["Left"] = (df[COL["attr"]] == "Yes").astype(int)

Employee_ID            0
Age                    0
Gender                 0
Department             0
Job_Level              0
Monthly_Income        30
Tenure_Years           0
Distance_From_Home     0
Overtime               0
Overtime_Hours         0
Satisfaction_Score    40
Attrition              0
dtype: int64
Duplicate Employee_ID: 8


In [48]:
# ---------- 3. Cleaning (with justification) ----------
# Income is right-skewed -> median is robust to outliers; impute within Department+Job_Level
df[COL["income"]] = df[COL["income"]].fillna(
    df.groupby([COL["dept"], COL["level"]])[COL["income"]].transform("median")
)
df[COL["income"]] = df[COL["income"]].fillna(df[COL["income"]].median())  # fallback

# Satisfaction is an ordinal score (e.g. 1-4) -> median keeps it a valid score value
df[COL["sat"]] = df[COL["sat"]].fillna(
    df.groupby(COL["dept"])[COL["sat"]].transform("median")
)
df[COL["sat"]] = df[COL["sat"]].fillna(df[COL["sat"]].median())
print("Missing after cleaning:\n", df[[COL["income"], COL["sat"]]].isna().sum())

Missing after cleaning:
 Monthly_Income        0
Satisfaction_Score    0
dtype: int64


In [49]:
# ---------- 4. Headcount, active, attrition rate ----------
total = len(df)
left = df["Left"].sum()
active = total - left
print(f"Total headcount: {total}\nActive: {active}\nLeft: {left}\n"
      f"Attrition rate: {left/total*100:.2f}%")

Total headcount: 1000
Active: 748
Left: 252
Attrition rate: 25.20%


In [50]:
# ---------- 5. Attrition by department ----------
dept_attr = pd.crosstab(df[COL["dept"]], df[COL["attr"]], normalize="index") * 100
print(dept_attr.round(2).sort_values("Yes", ascending=False))

Attrition      No    Yes
Department              
HR          68.29  31.71
Sales       68.77  31.23
Operations  75.82  24.18
Finance     78.57  21.43
R&D         80.41  19.59


In [51]:
# ---------- 6. Income & tenure: left vs stayed ----------
print(df.groupby(COL["attr"])[[COL["income"], COL["tenure"]]].mean().round(2))

           Monthly_Income  Tenure_Years
Attrition                              
No               55262.59          4.61
Yes              51672.54          3.86


In [52]:
# ---------- 7. Attrition by overtime ----------
ot_table = pd.crosstab(df[COL["ot_flag"]], df[COL["attr"]], normalize="index") * 100
ot_table["Headcount"] = df[COL["ot_flag"]].value_counts()
print(ot_table.round(2))

Attrition     No    Yes  Headcount
Overtime                          
No         81.88  18.12        701
Yes        58.19  41.81        299


In [53]:
# ---------- 8. Satisfaction by job level ----------
print(df.groupby(COL["level"])[COL["sat"]].agg(["mean", "count"]).round(2))

           mean  count
Job_Level             
Junior     2.89    342
Manager    3.12    111
Mid        2.86    344
Senior     2.87    203


In [54]:
# ---------- 9. Attrition by commute-distance band ----------
df["Distance_Band"] = pd.cut(df[COL["dist"]], bins=[0, 5, 10, 20, np.inf],
                             labels=["0-5 km", "5-10 km", "10-20 km", "20+ km"],
                             include_lowest=True)
dist_attr = df.groupby("Distance_Band", observed=True)["Left"].agg(
    Headcount="count", Attrition_Rate=lambda s: s.mean()*100).round(2)
print(dist_attr)

               Headcount  Attrition_Rate
Distance_Band                           
0-5 km               176           25.00
5-10 km              308           22.40
10-20 km             363           25.62
20+ km               153           30.07


In [55]:
# ---------- 10. Rank departments by avg overtime hours ----------
ot_rank = (df.groupby(COL["dept"])[COL["ot_hours"]].mean()
             .sort_values(ascending=False).round(2).to_frame("Avg_OT_Hours"))
ot_rank["Rank"] = range(1, len(ot_rank) + 1)
print(ot_rank)
# ok but it is not of the match of the

            Avg_OT_Hours  Rank
Department                    
Operations          7.63     1
Sales               6.12     2
R&D                 5.81     3
Finance             5.00     4
HR                  4.29     5


In [56]:
# ---------- 11. NumPy stats on income ----------
inc = df[COL["income"]].to_numpy()
print(f"Mean: {np.mean(inc):.2f} | Median: {np.median(inc):.2f} | Std: {np.std(inc, ddof=1):.2f}")

Mean: 54357.90 | Median: 44400.00 | Std: 31239.70


In [57]:
# ---------- 12. Outliers via IQR ----------
q1, q3 = np.percentile(inc, [25, 75])
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = df[(df[COL["income"]] < lo) | (df[COL["income"]] > hi)]
print(f"Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}, bounds=({lo:.0f}, {hi:.0f})")
print(f"Outliers: {len(outliers)}")
print(outliers[[COL["id"], COL["dept"], COL["level"], COL["income"]]].head(10))

Q1=31812, Q3=67577, IQR=35765, bounds=(-21835, 121224)
Outliers: 42
    Employee_ID Department Job_Level  Monthly_Income
0         E1113        R&D   Manager        132854.0
42        E1912    Finance   Manager        152101.0
54        E1664        R&D   Manager        162723.0
65        E1140        R&D       Mid        136738.0
107       E1285        R&D   Manager        150658.0
117       E1602    Finance       Mid        124242.0
135       E1446    Finance   Manager        137786.0
171       E1701        R&D   Manager        126721.0
173       E1908      Sales    Senior        121415.0
204       E1419    Finance   Manager        132582.5


In [58]:
# ---------- 13. Tenure bands ----------
df["Tenure_Band"] = pd.cut(df[COL["tenure"]], bins=[0, 2, 5, np.inf],
                           labels=["<2 yrs", "2-5 yrs", "5+ yrs"],
                           right=False, include_lowest=True)
ten_attr = df.groupby("Tenure_Band", observed=True)["Left"].agg(
    Headcount="count", Attrition_Rate=lambda s: s.mean()*100).round(2)
print(ten_attr)

             Headcount  Attrition_Rate
Tenure_Band                           
<2 yrs             193           41.45
2-5 yrs            475           21.05
5+ yrs             332           21.69


In [59]:
# ---------- 14. Department summary table ----------
dept_summary = df.groupby(COL["dept"]).agg(
    Headcount=(COL["id"], "count"),
    Avg_Salary=(COL["income"], "mean"),
    Avg_Tenure=(COL["tenure"], "mean"),
    Avg_OT_Hours=(COL["ot_hours"], "mean"),
    Attrition_Rate_Pct=("Left", lambda s: s.mean()*100),
).round(2).sort_values("Attrition_Rate_Pct", ascending=False)
print(dept_summary)

            Headcount  Avg_Salary  Avg_Tenure  Avg_OT_Hours  \
Department                                                    
HR                 82    50966.51        4.47          4.29   
Sales             285    52338.77        4.53          6.12   
Operations        244    46066.30        4.24          7.63   
Finance            98    63336.65        4.40          5.00   
R&D               291    61219.67        4.46          5.81   

            Attrition_Rate_Pct  
Department                      
HR                       31.71  
Sales                    31.23  
Operations               24.18  
Finance                  21.43  
R&D                      19.59  


In [60]:
# ---------- Bonus: correlation + save outputs ----------
print(df[[COL["income"], COL["sat"], COL["tenure"], COL["ot_hours"], "Left"]].corr().round(2))
dept_summary.to_csv("department_summary.csv")

                    Monthly_Income  Satisfaction_Score  Tenure_Years  \
Monthly_Income                1.00                0.03          0.05   
Satisfaction_Score            0.03                1.00          0.02   
Tenure_Years                  0.05                0.02          1.00   
Overtime_Hours               -0.01               -0.15         -0.01   
Left                         -0.05               -0.17         -0.11   

                    Overtime_Hours  Left  
Monthly_Income               -0.01 -0.05  
Satisfaction_Score           -0.15 -0.17  
Tenure_Years                 -0.01 -0.11  
Overtime_Hours                1.00  0.24  
Left                          0.24  1.00  
